# Energy Signature

This example demonstrates two building energy signature plots:

1. **Simple scatter** — daily mean outside temperature vs daily energy consumption, colored by season
2. **Proposed Energy Signature (PES)** — iterative algorithm (Eriksson et al., 2020) to find balance temperature, heat loss coefficient, standby and hot-water power, with regression lines

All plots are **interactive** — hover to see exact values.

In [ ]:
# Configure Plotly to render as HTML (required for Jupyter Book)
import plotly.io as pio
pio.renderers.default = "notebook_connected"

## Simple Energy Signature

A scatter plot of daily energy consumption against daily mean outside
temperature, colored by season. This is the most basic way to visualize
how a building's heating demand relates to outdoor conditions.

In [ ]:
import pandas as pd
from importlib import resources

# Load the bundled sample data
data_path = resources.files("pyedautils") / "data" / "bldg_engy_sig_simple.csv"
df_simple = pd.read_csv(data_path, sep=";")
df_simple.head()

### Outlier removal (IQR method)

Before plotting, we remove outliers from the `value` column using the
interquartile range (IQR) method. Values outside 1.5 × IQR below Q1
or above Q3 are dropped.

In [ ]:
q1 = df_simple["value"].quantile(0.25)
q3 = df_simple["value"].quantile(0.75)
iqr = q3 - q1

mask = df_simple["value"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
print(f"Removed {(~mask).sum()} outliers out of {len(df_simple)} rows")
df_simple = df_simple[mask]

In [ ]:
from pyedautils.plots import plot_energy_signature

fig = plot_energy_signature(
    data=df_simple,
    title="Building Energy Signature",
    xlab="Outside Temperature [°C]",
    ylab="Heating Energy [kWh/d]",
)
fig.show()

## Proposed Energy Signature (PES)

The PES method (Eriksson et al., 2020) iteratively determines:

- **T_b** — balance temperature: outdoor temperature below which heating is needed
- **Q_tot** — total heat loss coefficient [kW/K]
- **P_dhwc** — domestic hot water circulation demand (standby) [kW]
- **P_dhw** — domestic hot water demand [kW]

The algorithm uses hourly data with outdoor temperature, power consumption,
and room temperature.

In [ ]:
data_path = resources.files("pyedautils") / "data" / "bldg_engy_sig_pes.csv"
df_pes = pd.read_csv(data_path)
df_pes.head()

### Outlier removal (IQR method)

Remove outliers from the `power` column before computing PES parameters.

In [ ]:
q1 = df_pes["power"].quantile(0.25)
q3 = df_pes["power"].quantile(0.75)
iqr = q3 - q1

mask = df_pes["power"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
print(f"Removed {(~mask).sum()} outliers out of {len(df_pes)} rows")
df_pes = df_pes[mask]

### Compute PES parameters

You can compute the PES parameters directly using `compute_pes`:

In [ ]:
from pyedautils.energy_signature import compute_pes

result = compute_pes(df_pes)
print(f"Balance temperature:      T_b    = {result.tb:.1f} °C")
print(f"Heat loss coefficient:    Q_tot  = {result.q_tot:.4f} kW/K")
print(f"DHWC demand (standby):    P_dhwc = {result.p_dhwc:.4f} kW")
print(f"DHW demand:               P_dhw  = {result.p_dhw:.4f} kW")
print(f"Internal heat gains:      P_ihg  = {result.p_ihg:.4f} kW")

### PES Plot

The plot shows daily mean power vs outdoor temperature, with three
characteristic lines: the heating line (slope = Q_tot), the base load
line (P_dhw + P_dhwc), and the DHWC line (P_dhwc).

In [ ]:
from pyedautils.plots import plot_energy_signature_pes

fig = plot_energy_signature_pes(
    data=df_pes,
    title="Proposed Energy Signature",
    xlab="Outside Temperature [°C]",
    ylab="Power [kW]",
)
fig.show()

### With internal heat gains

If internal heat gains (people, appliances, solar) are known, pass
them via the `p_ihg` parameter to improve accuracy:

In [ ]:
fig = plot_energy_signature_pes(
    data=df_pes,
    p_ihg=0.54,
    title="PES with Internal Heat Gains",
)
fig.show()